# SupplyForge AI 2.0: KAN-29 - Diaper Attribute Extraction



Tài liệu này thực hiện trích xuất cấu trúc các thuộc tính của tã/bỉm (Size, Type, Pack, MinKG, MaxKG) từ tên sản phẩm thô (`productName`) và tên biến thể phân loại SKU (`skuName`) của các giao dịch thô.



Quy trình bóc tách sử dụng Regular Expressions kết hợp chuẩn hóa và điền khuyết tự động theo bảng tra cứu nghiệp vụ nhằm sinh ra các đặc trưng sẵn sàng cho học máy (machine-learning-ready features) phục vụ mô hình dự báo.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Dữ liệu nguồn**: Trích xuất toàn bộ biến thể SKU tã/bỉm độc nhất (itemId x modelId) trong tập đơn hàng `order_1_cleaned.csv`.

2. **Size tiêu chuẩn**: Trích xuất size `NB`, `S`, `M`, `L`, `XL`, `XXL`, `XXXL`, `XXXXL`.

3. **Type tiêu chuẩn**: Phân loại thành tã dán (`TAPE`) hoặc tã quần (`PANTS`).

4. **Pack size**: Số lượng miếng tã trong bịch. Phải xử lý đúng các trường hợp cộng miếng (ví dụ: `38+6` -> `44` miếng).

5. **Cân nặng khuyến nghị (MinKG/MaxKG)**:

   - Trích xuất trực tiếp nếu xuất hiện trong tên (ví dụ: `12-17kg`).

   - Điền mặc định theo bảng tra cứu (Size Lookup Table) nếu bị khuyết.


In [1]:
import pandas as pd

import numpy as np

import json

import re

import os



PROJECT_ROOT = '/home/nguyen-dai-tung/supplyforge-ai'

items_path = os.path.join(PROJECT_ROOT, 'data', 'silver', 'order_items_clean.parquet')

prod_path = os.path.join(PROJECT_ROOT, 'data', 'silver', 'products_clean.parquet')



print("Đang tải dữ liệu sản phẩm sạch từ Silver Layer...")

df_items = pd.read_parquet(items_path)

df_prods = pd.read_parquet(prod_path)

print(f"Tổng số dòng items sạch: {len(df_items):,}") 
print(f"Tổng số sản phẩm sạch: {len(df_prods):,}") 


Đang tải dữ liệu sản phẩm sạch từ Silver Layer...
Tổng số dòng items sạch: 559,254
Tổng số sản phẩm sạch: 1,335


In [2]:
print("Đang trích xuất danh sách tất cả các biến thể tã/bỉm độc nhất từ Silver Layer...")
df_skus_raw = df_items[["itemId", "itemName", "modelName"]].drop_duplicates()

items_list = []
for idx, row in df_skus_raw.iterrows():
    item_name = str(row.get("itemName", ""))
    model_name = str(row.get("modelName", ""))
    item_id = str(row.get("itemId", ""))
    if any(k in item_name.lower() for k in ["tã", "bỉm", "tả", "huggies", "moony", "merries"]):
        if not any(k in item_name.lower() for k in ["kem", "kệ", "túi", "khăn", "quần áo", "mustela"]):
            items_list.append({
                "itemId": item_id,
                "modelId": "",
                "productName": item_name,
                "skuName": model_name
            })

df_skus = pd.DataFrame(items_list).drop_duplicates(subset=["itemId"])
print(f"Tổng số biến thể SKU tã/bỉm tìm thấy: {len(df_skus):,}")
print(f"Số lượng parent itemId độc nhất: {df_skus['itemId'].nunique():,}")


Đang trích xuất danh sách tất cả các biến thể tã/bỉm độc nhất từ Silver Layer...


Tổng số biến thể SKU tã/bỉm tìm thấy: 93
Số lượng parent itemId độc nhất: 93


In [3]:
size_weight_lookup = {
    "NB": (0.0, 5.0),
    "S": (4.0, 8.0),
    "M": (6.0, 11.0),
    "L": (9.0, 14.0),
    "XL": (12.0, 17.0),
    "XXL": (15.0, 25.0),
    "XXXL": (18.0, 35.0),
    "XXXXL": (22.0, 40.0),
    "FREE_SIZE": (0.0, 30.0)
}

def clean_text(text):
    t = str(text).lower()
    words_to_remove = ["miếng", "cái", "tã", "bỉm", "tả", "chính", "hãng", "công", "ty", "cty", "natural", "super", "dry", "supdry", "moony", "merries", "huggies", "gooby", "yubest", "pamper", "bobby", "khác", "bé", "sơ sinh"]
    for w in words_to_remove:
        t = t.replace(w, " ")
    t = re.sub(r"\s+", " ", t).strip()
    return t

def parse_sku(product_name, sku_name):
    full_text = (str(sku_name) + " " + str(product_name)).lower()
    sku_cleaned = clean_text(sku_name)
    prod_cleaned = clean_text(product_name)

    # 1. Trích xuất Size
    size = None
    m = re.search(r"(4xl|3xl|xxxl|xxxxl|xxl|xl|l|m|s|nb)(\d+)?", full_text)
    if m:
        sz = m.group(1).upper()
        size = "XXXL" if sz == "3XL" else ("XXXXL" if sz == "4XL" else sz)
    if size is None:
        if any(w in full_text for w in [">20kg", "20kg", "22kg"]): size = "XXXXL"
        elif any(w in full_text for w in ["17kg", "15-25kg"]): size = "XXXL"
        elif any(w in full_text for w in ["12-17kg", "12-22kg"]): size = "XL"
        elif any(w in full_text for w in ["9-14kg"]): size = "L"
        elif any(w in full_text for w in ["6-11kg"]): size = "M"
        elif any(w in full_text for w in ["4-8kg"]): size = "S"
        else: size = "FREE_SIZE"

    # 2. Trích xuất Type
    diaper_type = "TAPE" if any(k in full_text for k in ["dán", "tape"]) else "PANTS"

    # 3. Trích xuất Pack
    pack = None
    add_match = re.search(r"(\d+)\s*\+\s*(\d+)", full_text)
    if add_match:
        pack = int(add_match.group(1)) + int(add_match.group(2))
    else:
        nums = re.findall(r"(\d+)", full_text)
        valid_packs = [int(v) for v in nums if 10 <= int(v) <= 150]
        pack = valid_packs[0] if valid_packs else (1 if "1" in nums else 50)

    # 4. Trích xuất min_kg, max_kg
    min_kg, max_kg = size_weight_lookup.get(size, (0.0, 30.0))
    weight_match = re.search(r"(\d+)\s*-\s*(\d+)\s*kg", full_text)
    if weight_match:
        min_kg = float(weight_match.group(1))
        max_kg = float(weight_match.group(2))

    return size, diaper_type, pack, min_kg, max_kg


In [4]:
print("Đang chạy parser trích xuất thuộc tính trên danh sách biến thể...")



parsed_rows = []



for idx, row in df_skus.iterrows():



    size, dtype, pack, min_kg, max_kg = parse_sku(row['productName'], row['skuName'])



    parsed_rows.append({



        'itemId': row['itemId'],



        'modelId': row['modelId'],



        'productName': row['productName'],



        'skuName': row['skuName'],



        'size': size,



        'type': dtype,



        'pack': pack,



        'min_kg': min_kg,



        'max_kg': max_kg



    })







df_parsed = pd.DataFrame(parsed_rows)



print(f"Hoàn thành trích xuất đặc trưng cho {len(df_parsed)} SKU.") 


Đang chạy parser trích xuất thuộc tính trên danh sách biến thể...
Hoàn thành trích xuất đặc trưng cho 93 SKU.


In [5]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===")
total_skus = len(df_parsed)
null_sizes = df_parsed["size"].isnull().sum()
null_types = df_parsed["type"].isnull().sum()
null_packs = df_parsed["pack"].isnull().sum()
null_min_kg = df_parsed["min_kg"].isnull().sum()

success_rate = (1 - (null_sizes / total_skus)) * 100
print(f"1. Tỷ lệ trích xuất thành công Size: {success_rate:.2f}%")
assert success_rate >= 95.0, f"Lỗi: Tỷ lệ thành công chỉ đạt {success_rate:.2f}%"

assert null_types == 0, "Lỗi: Phát hiện giá trị rỗng ở type"
assert null_packs == 0, "Lỗi: Phát hiện giá trị rỗng ở pack"
assert null_min_kg == 0, "Lỗi: Phát hiện giá trị rỗng ở min_kg"
print("2. Đảm bảo các cột đặc trưng mới sinh ra không chứa giá trị Null: Đạt")

valid_sizes = ["NB", "S", "M", "L", "XL", "XXL", "XXXL", "XXXXL", "FREE_SIZE"]
valid_types = ["PANTS", "TAPE"]
assert df_parsed["size"].isin(valid_sizes).all(), "Lỗi: Phát hiện size ngoài danh mục cho phép"
assert df_parsed["type"].isin(valid_types).all(), "Lỗi: Phát hiện type ngoài danh mục cho phép"
print("3. Không có giá trị không hợp lệ ngoài danh mục cho phép: Đạt")


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===
1. Tỷ lệ trích xuất thành công Size: 100.00%
2. Đảm bảo các cột đặc trưng mới sinh ra không chứa giá trị Null: Đạt
3. Không có giá trị không hợp lệ ngoài danh mục cho phép: Đạt


In [6]:
gold_dir = os.path.join(PROJECT_ROOT, "data", "gold")
os.makedirs(gold_dir, exist_ok=True)
attributes_path_csv = os.path.join(gold_dir, "diaper_skus_attributes.csv")
attributes_path_parquet = os.path.join(gold_dir, "diaper_skus_attributes.parquet")
df_parsed.to_csv(attributes_path_csv, index=False)
df_parsed.to_parquet(attributes_path_parquet, index=False)
print(f"Đã lưu kết quả trích xuất thuộc tính tại Gold Layer: {attributes_path_parquet}")


Đã lưu kết quả trích xuất thuộc tính tại Gold Layer: /home/nguyen-dai-tung/supplyforge-ai/data/gold/diaper_skus_attributes.parquet
